In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
import os
import seaborn as sns

In [8]:
def nse(y_true, y_pred):
    nse_np =1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
    return float(nse_np)

def pbias(y_true, y_pred):
    pbias_value = 100 * np.sum(y_true - y_pred) / np.sum(y_true)
    return float(pbias_value)

def kge(y_true, y_pred):
    # Calculate the Pearson correlation coefficient (r)
    r, _ = pearsonr(y_true, y_pred)
    
    # Calculate the mean of the observed and predicted values
    mu_true = np.mean(y_true)
    mu_pred = np.mean(y_pred)
    
    # Calculate the standard deviation of the observed and predicted values
    sigma_true = np.std(y_true)
    sigma_pred = np.std(y_pred)
    
    # Compute the KGE
    kge_value = 1 - np.sqrt((r - 1)**2 + (sigma_pred / sigma_true - 1)**2 + (mu_pred / mu_true - 1)**2)
    
    return kge_value

In [9]:
input_file_name_list = ['1_hsg_imputMF', '2_dsk_imputMF', '3_xhl_imputMF', '4_slglk_imputMF', '5_kq_imputMF',
                        '6_wlwt_imputMF', '7_tgzlk_imputMF']

In [10]:
os.chdir('F:\\geodata\\river_runoff_obs')
GCM_name = "MRI-ESM2-0"
"MRI-ESM2-0"
'INM-CM4-8'
'BCC-CSM2-MR'
'INM-CM5-0'
print(GCM_name)

MRI-ESM2-0


In [ ]:
from datetime import datetime# Get current time
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
txtbook_path = f'output\\XGBoost_daily_note_{current_time}_fixed.txt'
print(txtbook_path)
for GCM_name in [ 'FGOALS-g3', 'MPI-ESM1-2-HR', 'EC-Earth3','BCC-CSM2-MR', 'MRI-ESM2-0', 'INM-CM5-0', 'INM-CM4-8']:
    for input_file_name in input_file_name_list:
        abbre = input_file_name.split('_')[1]    
        for scenario in ['ssp126', 'ssp245', 'ssp370', 'ssp585']:
            name = f'{input_file_name}_{GCM_name}_{scenario}_r1i1p1f1_daily'          
       
        
            month_df = pd.read_csv(f"F:\\geodata\\river_runoff_obs\\XGBoost_mon\\{name.replace("daily", "mon")}_project.csv")
            daily_df = pd.read_csv(f'F:\\geodata\\river_runoff_obs\\XGBoost_daily\\{name}_project.csv')
            # Convert the 'time' column to datetime format
            month_df['time'] = pd.to_datetime(month_df['time'])
            daily_df['time'] = pd.to_datetime(daily_df['time'])
            daily_df['date'] = daily_df['time']
            
            # Extract year and month from the 'time' column
            month_df['year_month'] = month_df['time'].dt.to_period('M')
            daily_df['year_month'] = daily_df['time'].dt.to_period('M')
            # Extract the year from 'year_month' and add it as a new column
            daily_df['year'] = daily_df.year_month.dt.year
            merged_df = pd.merge(daily_df, month_df, on='year_month', how='left', suffixes=('_daily', '_month'))
            # Group by 'year' and calculate the minimum of 'runoff' and 'ep' for each year
            annual_min = merged_df.groupby('year').agg(min_runoff=('runoff_projection_daily', 'min'), min_ep=('evaporation_projection_daily', 'min')).reset_index()
            month_mean = merged_df.groupby('year_month').agg(mean_runoff=('runoff_projection_daily', 'mean'), mean_ep=('evaporation_projection_daily', 'mean')).reset_index()
            
            merged_df= merged_df.merge(annual_min,on = 'year', how='left')
            merged_df= merged_df.merge(month_mean,on = 'year_month', how='left')
            
            # merged_df['runoff_fixed'] = (merged_df.runoff_projection_daily-merged_df.min_runoff)/(merged_df.mean_runoff-merged_df.min_runoff)*(merged_df.runoff_projection_month-merged_df.min_runoff) + merged_df.min_runoff
            # merged_df['ep_fixed'] = (merged_df.evaporation_projection_daily-merged_df.min_ep)/(merged_df.mean_ep-merged_df.min_ep)*(merged_df.evaporation_projection_month/30-merged_df.min_ep) + merged_df.min_ep
            
            merged_df['runoff_fixed'] = (merged_df.runoff_projection_daily-merged_df.min_runoff)/(merged_df.mean_runoff-merged_df.min_runoff)*(merged_df.runoff_projection_month-merged_df.min_runoff) + merged_df.min_runoff
            merged_df['ep_fixed'] = (merged_df.evaporation_projection_daily-merged_df.min_ep)/(merged_df.mean_ep-merged_df.min_ep)*(merged_df.evaporation_projection_month/30-merged_df.min_ep) + merged_df.min_ep
            
            merged_df.to_csv('output\\'+name+'_fixed.csv', index=False)
            historical_df = merged_df.dropna(subset=['dis_daily'])
            validation_df = historical_df.tail(int(len(historical_df)*0.3)) #here the test rate is 0.3
            validation_df.set_index('time_daily', inplace=True)
            
            print(scenario,'NSE of runoff: ',nse(validation_df['dis_daily'], validation_df['runoff_fixed']))
            print(scenario,'KGE of runoff: ',kge(validation_df['dis_daily'], validation_df['runoff_fixed']))
            print(scenario,'Pias of runoff:',pbias(validation_df['dis_daily'], validation_df['runoff_fixed']))
        
            # # validation_df[['dis_daily','runoff_projection_daily','runoff_projection_month','dis_month','runoff_fixed']].plot(figsize=(20,16))
            # plt.scatter(validation_df['dis_daily'],validation_df['runoff_fixed'],c='b')
            # plt.show()
            
            with open(txtbook_path, 'a') as file:
                file.write(f"Name: {name}, Scenario: {scenario},NSE Runoff: {nse(validation_df['dis_daily'], validation_df['runoff_fixed'])}, KGE Runoff: {kge(validation_df['dis_daily'], validation_df['runoff_fixed'])},Pbias: {pbias(validation_df['dis_daily'], validation_df['runoff_fixed'])}\n")
            

output\XGBoost_daily_note_2025-01-04_21-27-19_fixed.txt
ssp126 NSE of runoff:  0.643722873974228
ssp126 KGE of runoff:  0.6842939175236271
ssp126 Pias of runoff: 5.085701487156645
ssp245 NSE of runoff:  0.5720043049774175
ssp245 KGE of runoff:  0.6580603230502271
ssp245 Pias of runoff: 2.83384439612836
ssp370 NSE of runoff:  0.5861996972360561
ssp370 KGE of runoff:  0.655646837120393
ssp370 Pias of runoff: 4.151117449505635
ssp585 NSE of runoff:  0.6065067407431108
ssp585 KGE of runoff:  0.6797527164782431
ssp585 Pias of runoff: 3.540321495479804
ssp126 NSE of runoff:  0.6876434768199998
ssp126 KGE of runoff:  0.8037716826752764
ssp126 Pias of runoff: -4.616441424993766
ssp245 NSE of runoff:  0.6960719058310445
ssp245 KGE of runoff:  0.8067016656626136
ssp245 Pias of runoff: -4.2606909244170605
ssp370 NSE of runoff:  0.680529384984953
ssp370 KGE of runoff:  0.7912146204726067
ssp370 Pias of runoff: -3.650170019708758
ssp585 NSE of runoff:  0.6663676009433588
ssp585 KGE of runoff:  0.79